# 02 - Exploratory Data Analysis

This notebook loads processed data and performs comprehensive EDA:
- Revenue distributions
- Monthly trends
- Category, channel, and country analysis
- Correlation matrix
- Display of generated figures from outputs/figures/

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")

print("Libraries loaded successfully.")

## 2. Load Processed Data

In [ ]:
processed_path = Path('../data/processed')

orders = pd.read_csv(processed_path / 'orders_processed.csv')
orders["order_date"] = pd.to_datetime(orders["order_date"])

customers = pd.read_csv(processed_path / 'customers_processed.csv')

print(f'Orders: {orders.shape}')
print(f'Customers: {customers.shape}')
print(f'\nOrder columns: {list(orders.columns)}')

## 3. Revenue Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of order amounts
axes[0].hist(orders["revenue"], bins=50, color="steelblue", edgecolor="white", alpha=0.7)
axes[0].set_title("Distribution of Order Revenue")
axes[0].set_xlabel("Revenue")
axes[0].set_ylabel("Frequency")

# Log-transformed histogram
positive_revenue = orders[orders["revenue"] > 0]["revenue"]
axes[1].hist(np.log1p(positive_revenue), bins=50, color="coral", edgecolor="white", alpha=0.7)
axes[1].set_title("Distribution of Log(Revenue + 1)")
axes[1].set_xlabel("Log Revenue")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

## 4. Monthly Revenue Trends

In [ ]:
orders['year_month'] = orders['order_date'].dt.to_period('M')
monthly_revenue = orders.groupby('year_month').agg(
    revenue=('revenue', 'sum'),
    orders=('order_id', 'count'),
    avg_order_value=('revenue', 'mean')
).reset_index()
monthly_revenue['year_month_str'] = monthly_revenue['year_month'].astype(str)

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Revenue trend
axes[0].plot(monthly_revenue["year_month_str"], monthly_revenue["revenue"], marker="o", color="steelblue")
axes[0].set_title("Monthly Total Revenue")
axes[0].set_ylabel("Revenue")
axes[0].tick_params(axis="x", rotation=45)

# Order count trend
axes[1].bar(monthly_revenue["year_month_str"], monthly_revenue["orders"], color="coral", alpha=0.7)
axes[1].set_title("Monthly Order Count")
axes[1].set_ylabel("Number of Orders")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## 5. Category Analysis

In [ ]:
if 'category' in orders.columns:
    category_revenue = orders.groupby('category').agg(
        revenue=('revenue', 'sum'),
        orders=('order_id', 'count'),
        avg_revenue=('revenue', 'mean')
    ).sort_values('revenue', ascending=False)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    category_revenue["revenue"].plot(kind="bar", ax=axes[0], color="steelblue")
    axes[0].set_title("Revenue by Category")
    axes[0].set_ylabel("Revenue")
    axes[0].tick_params(axis="x", rotation=45)

    category_revenue["orders"].plot(kind="bar", ax=axes[1], color="coral")
    axes[1].set_title("Order Count by Category")
    axes[1].set_ylabel("Orders")
    axes[1].tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.show()
else:
    print("No category column found in orders data.")

## 6. Channel Analysis

In [ ]:
if 'channel' in orders.columns:
    channel_revenue = orders.groupby('channel').agg(
        revenue=('revenue', 'sum'),
        orders=('order_id', 'count'),
        avg_revenue=('revenue', 'mean')
    ).sort_values('revenue', ascending=False)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].pie(channel_revenue["revenue"], labels=channel_revenue.index, autopct="%1.1f%%")
    axes[0].set_title("Revenue Share by Channel")

    channel_revenue["avg_revenue"].plot(kind="bar", ax=axes[1], color="seagreen")
    axes[1].set_title("Average Order Value by Channel")
    axes[1].set_ylabel("Avg Revenue")
    axes[1].tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.show()
else:
    print("No channel column found in orders data.")

## 7. Country Analysis

In [ ]:
if 'country' in orders.columns:
    country_revenue = orders.groupby('country').agg(
        revenue=('revenue', 'sum'),
        orders=('order_id', 'count'),
        avg_revenue=('revenue', 'mean')
    ).sort_values('revenue', ascending=False).head(15)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    country_revenue["revenue"].plot(kind="bar", ax=axes[0], color="steelblue")
    axes[0].set_title("Top 15 Countries by Revenue")
    axes[0].set_ylabel("Revenue")
    axes[0].tick_params(axis="x", rotation=45)

    country_revenue["orders"].plot(kind="bar", ax=axes[1], color="coral")
    axes[1].set_title("Top 15 Countries by Order Count")
    axes[1].set_ylabel("Orders")
    axes[1].tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.show()
else:
    print("No country column found in orders data.")

## 8. Correlation Matrix

In [ ]:
numeric_cols = orders.select_dtypes(include=[np.number]).columns.tolist()

if len(numeric_cols) > 1:
    corr_matrix = orders[numeric_cols].corr()

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f', ax=ax)
    ax.set_title('Correlation Matrix - Numeric Features')
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numeric columns for correlation matrix.")

## 9. Display Generated Figures

In [ ]:
figures_path = Path('../outputs/figures')

generated_figs = list(figures_path.glob('*.png')) if figures_path.exists() else []

if generated_figs:
    print(f"Found {len(generated_figs)} generated figures:\n")
    for fig_path in sorted(generated_figs):
        print(f"  - {fig_path.name}")
else:
    print("No generated PNG figures found in outputs/figures/.")

## 10. Summary Statistics

In [ ]:
print('EDA Summary')
print("=" * 60)
print(f"Total orders: {orders.shape[0]:,}")
print(f"Total revenue: ")
print(f"Average order value: ")
print(f"Median order value: ")
print(f"Date range: {orders["order_date"].min()} to {orders["order_date"].max()}")
if "category" in orders.columns:
    print(f"Unique categories: {orders["category"].nunique()}")
if "channel" in orders.columns:
    print(f"Unique channels: {orders["channel"].nunique()}")
if "country" in orders.columns:
    print(f"Unique countries: {orders["country"].nunique()}")